# 02 Validation

This notebook provides representative validation analyses for the released LLM-derived familiarity estimates.

It includes two parts:

1. **Alignment with human familiarity norms**
2. **Prediction of lexical decision reaction times**

## Required local files

Before running this notebook, please download the required external datasets from the original sources listed in the repository README and place them under:

- `../External_Data/`

### Released familiarity datasets
These are already included in the repository under:
- `../Data/2 norms/`

### External human familiarity datasets
For human familiarity validation, the table only needs to contain:
- one column for lexical items
- one column for familiarity ratings

### External lexical decision dataset
For lexical decision modeling, the table should contain at least:
- one column for lexical items
- one column for lexical decision reaction times (`zRT`)
- one column for error rate (`ERR`) or accuracy (`ACC`)

### Optional benchmark dataset
For optional benchmark comparisons, the table should contain at least:
- one column for lexical items
- one column for word frequency
- one column for contextual diversity

In [ ]:
## Step 1. Import packages and define helper functions

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import statsmodels.api as sm

# -----------------------------
# Helper functions
# -----------------------------
def standardize_word_column(df, word_col, new_word_col="WORD"):
    df = df.copy()
    df[new_word_col] = df[word_col].astype(str).str.strip()
    return df

def run_pearson(df, x_col, y_col):
    sub = df[[x_col, y_col]].dropna()
    r, p = pearsonr(sub[x_col], sub[y_col])
    return {
        "n": len(sub),
        "r": r,
        "p": p
    }

def run_single_predictor_regression(df, predictor_col, outcome_col="zRT"):
    sub = df[[predictor_col, outcome_col]].dropna().copy()
    X = sm.add_constant(sub[predictor_col])
    y = sub[outcome_col]
    model = sm.OLS(y, X).fit()
    return {
        "predictor": predictor_col,
        "n": len(sub),
        "beta": model.params[predictor_col],
        "p": model.pvalues[predictor_col],
        "r2": model.rsquared
    }, model

## Step 2. Define file paths and variable names

Please adjust the settings below according to the dataset you want to validate.

### Notes
- For single-character validation, use the character-level familiarity file and the Liu et al. human dataset.
- For multi-character validation, use the word- or expression-level familiarity file and the Su et al. human dataset.
- `length_filter` can be set to `1`, `2`, `3`, `4`, or `None`.

In [ ]:
# -----------------------------
# Output directory
# -----------------------------
output_dir = Path("./output")
output_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# LLM familiarity dataset
# -----------------------------
llm_file = Path("../Data/2 norms/norms_gpt4o_word_prompt.xlsx")
llm_word_col = "WORD"
llm_length_col = "Length"
llm_fam_col = "GPT_FAM_probs"

# Optional length restriction:
# 1 = single-character
# 2 = two-character
# 3 = three-character
# 4 = four-character
# None = no restriction
length_filter = 2

# -----------------------------
# Human familiarity dataset
# -----------------------------
# Example:
# ../External_Data/human_fam_liu.xlsx
# ../External_Data/human_fam_su.xlsx
human_file = Path("../External_Data/human_fam_su.xlsx")
human_word_col = "word"
human_fam_col = "FAM"

# -----------------------------
# Lexical decision dataset
# -----------------------------
ldt_file = Path("../External_Data/meld_sch.xlsx")
ldt_word_col = "word"
ldt_rt_col = "zRT"

# If your file already has ACC, fill in ldt_acc_col.
# If it only has ERR, fill in ldt_err_col and leave ldt_acc_col = None.
ldt_acc_col = None
ldt_err_col = "ERR"

# Is ERR recorded as percentage points (e.g., 12.5)?
# If yes, accuracy will be computed as 100 - ERR.
err_is_percent = True

# Only keep words with accuracy above this threshold
acc_threshold = 85.0 if err_is_percent else 0.85

# -----------------------------
# Optional benchmark dataset
# -----------------------------
# Example:
# ../External_Data/subtlex_ch.xlsx
benchmark_file = Path("../External_Data/subtlex_ch.xlsx")
benchmark_word_col = "word"
benchmark_wf_col = "WF"
benchmark_cd_col = "CD"

run_benchmarks = True

## Step 3. Load the released LLM familiarity dataset

In [ ]:
if not llm_file.exists():
    raise FileNotFoundError(f"LLM familiarity file not found: {llm_file}")

df_llm = pd.read_excel(llm_file)
df_llm = standardize_word_column(df_llm, llm_word_col, "WORD")

required_llm_cols = [llm_length_col, llm_fam_col, "WORD"]
missing_llm_cols = [col for col in required_llm_cols if col not in df_llm.columns]
if missing_llm_cols:
    raise ValueError(f"Missing required columns in LLM dataset: {missing_llm_cols}")

df_llm = df_llm.rename(columns={
    llm_length_col: "Length",
    llm_fam_col: "LLM_FAM"
})

if length_filter is not None:
    df_llm = df_llm[df_llm["Length"] == length_filter].copy()

print(f"LLM dataset loaded: {len(df_llm)} rows after length filtering.")
df_llm[["WORD", "Length", "LLM_FAM"]].head()

## Part A. Alignment with human familiarity norms

### Step A1. Load and standardize the human familiarity dataset

In [ ]:
if not human_file.exists():
    raise FileNotFoundError(f"Human familiarity file not found: {human_file}")

df_human = pd.read_excel(human_file)
df_human = standardize_word_column(df_human, human_word_col, "WORD")

required_human_cols = [human_fam_col, "WORD"]
missing_human_cols = [col for col in required_human_cols if col not in df_human.columns]
if missing_human_cols:
    raise ValueError(f"Missing required columns in human familiarity dataset: {missing_human_cols}")

df_human = df_human.rename(columns={
    human_fam_col: "HUMAN_FAM"
})

print(f"Human familiarity dataset loaded: {len(df_human)} rows.")
df_human[["WORD", "HUMAN_FAM"]].head()

### Step A2. Merge the LLM familiarity data with the human familiarity data

In [ ]:
df_human_merge = df_llm.merge(
    df_human[["WORD", "HUMAN_FAM"]],
    on="WORD",
    how="inner"
)

print(f"Merged rows for human familiarity validation: {len(df_human_merge)}")
df_human_merge[["WORD", "LLM_FAM", "HUMAN_FAM"]].head()

### Step A3. Compute the Pearson correlation

In [ ]:
human_corr_result = run_pearson(df_human_merge, "LLM_FAM", "HUMAN_FAM")
human_corr_result

### Step A4. Save the merged table and the correlation result

In [ ]:
human_merge_output = output_dir / "validation_human_familiarity_merged.xlsx"
human_result_output = output_dir / "validation_human_familiarity_result.csv"

df_human_merge.to_excel(human_merge_output, index=False)
pd.DataFrame([human_corr_result]).to_csv(human_result_output, index=False)

print(f"Saved merged table to: {human_merge_output}")
print(f"Saved correlation result to: {human_result_output}")

## Part B. Predicting lexical decision reaction times

### Step B1. Load and standardize the lexical decision dataset

In [ ]:
if not ldt_file.exists():
    raise FileNotFoundError(f"LDT file not found: {ldt_file}")

df_ldt = pd.read_excel(ldt_file)
df_ldt = standardize_word_column(df_ldt, ldt_word_col, "WORD")

required_ldt_cols = [ldt_rt_col, "WORD"]
missing_ldt_cols = [col for col in required_ldt_cols if col not in df_ldt.columns]
if missing_ldt_cols:
    raise ValueError(f"Missing required columns in lexical decision dataset: {missing_ldt_cols}")

df_ldt = df_ldt.rename(columns={
    ldt_rt_col: "zRT"
})

if ldt_acc_col is not None:
    df_ldt = df_ldt.rename(columns={ldt_acc_col: "ACC"})
elif ldt_err_col is not None:
    if ldt_err_col not in df_ldt.columns:
        raise ValueError(f"ERR column not found: {ldt_err_col}")
    if err_is_percent:
        df_ldt["ACC"] = 100 - df_ldt[ldt_err_col]
    else:
        df_ldt["ACC"] = 1 - df_ldt[ldt_err_col]
else:
    raise ValueError("Please provide either ldt_acc_col or ldt_err_col.")

df_ldt = df_ldt[df_ldt["ACC"] > acc_threshold].copy()

print(f"LDT dataset loaded: {len(df_ldt)} rows after ACC filtering.")
df_ldt[["WORD", "zRT", "ACC"]].head()

### Step B2. Merge the LLM familiarity data with the lexical decision data

In [ ]:
df_ldt_merge = df_llm.merge(
    df_ldt[["WORD", "zRT", "ACC"]],
    on="WORD",
    how="inner"
)

print(f"Merged rows for LDT modeling: {len(df_ldt_merge)}")
df_ldt_merge[["WORD", "LLM_FAM", "zRT", "ACC"]].head()

### Step B3. Run a single-predictor regression using LLM-derived familiarity

In [ ]:
ldt_llm_result, ldt_llm_model = run_single_predictor_regression(
    df_ldt_merge,
    predictor_col="LLM_FAM",
    outcome_col="zRT"
)

ldt_llm_result

### Step B4. Save the merged table and the LDT regression result

In [ ]:
ldt_merge_output = output_dir / "validation_ldt_llm_merged.xlsx"
ldt_result_output = output_dir / "validation_ldt_llm_result.csv"

df_ldt_merge.to_excel(ldt_merge_output, index=False)
pd.DataFrame([ldt_llm_result]).to_csv(ldt_result_output, index=False)

print(f"Saved merged table to: {ldt_merge_output}")
print(f"Saved regression result to: {ldt_result_output}")

## Optional benchmark comparison

This section is optional. If benchmark files are available, the notebook can also compare LLM-derived familiarity with:
- human familiarity
- lexical frequency
- contextual diversity

All benchmark analyses below are single-predictor regressions on lexical decision reaction times.

In [ ]:
benchmark_results = []

# -----------------------------
# Human familiarity benchmark
# -----------------------------
if "HUMAN_FAM" in df_human_merge.columns:
    df_ldt_human = df_ldt.merge(
        df_human_merge[["WORD", "HUMAN_FAM"]],
        on="WORD",
        how="inner"
    )
    if len(df_ldt_human) > 0:
        result, _ = run_single_predictor_regression(
            df_ldt_human,
            predictor_col="HUMAN_FAM",
            outcome_col="zRT"
        )
        benchmark_results.append(result)

# -----------------------------
# Frequency/CD benchmark
# -----------------------------
if run_benchmarks:
    if benchmark_file.exists():
        df_bench = pd.read_excel(benchmark_file)
        df_bench = standardize_word_column(df_bench, benchmark_word_col, "WORD")

        required_bench_cols = [benchmark_wf_col, benchmark_cd_col, "WORD"]
        missing_bench_cols = [col for col in required_bench_cols if col not in df_bench.columns]
        if missing_bench_cols:
            raise ValueError(f"Missing required columns in benchmark dataset: {missing_bench_cols}")

        df_bench = df_bench.rename(columns={
            benchmark_wf_col: "WF",
            benchmark_cd_col: "CD"
        })

        df_ldt_bench = df_ldt.merge(
            df_bench[["WORD", "WF", "CD"]],
            on="WORD",
            how="inner"
        )

        if len(df_ldt_bench) > 0:
            result_wf, _ = run_single_predictor_regression(
                df_ldt_bench,
                predictor_col="WF",
                outcome_col="zRT"
            )
            benchmark_results.append(result_wf)

            result_cd, _ = run_single_predictor_regression(
                df_ldt_bench,
                predictor_col="CD",
                outcome_col="zRT"
            )
            benchmark_results.append(result_cd)

benchmark_results_df = pd.DataFrame(benchmark_results)
benchmark_results_df

### Step B5. Save the optional benchmark comparison results

In [ ]:
if len(benchmark_results_df) > 0:
    benchmark_output = output_dir / "validation_ldt_benchmark_results.csv"
    benchmark_results_df.to_csv(benchmark_output, index=False)
    print(f"Saved benchmark comparison results to: {benchmark_output}")
else:
    print("No optional benchmark results were generated.")